### Import packages

In [ ]:
from pathlib import Path
import geopandas as gpd
import pyproj
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from matplotlib.patches import Patch
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import scipy.interpolate

import fiona
# Import for making the titles on the figures the same
import matplotlib as mpl

### Set up base paths

#### Base path and network path

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis")
direct_damages_folder = base_path / "Results/direct_damages_summary"
networks_catchments_folder = base_path / "Processed_data/networks/networks_catchments_intersections"
aggregated_summaries = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/aggregated_summaries")


In [ ]:
# Define an output directory for the joined files
output_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/joined_networks_catchments_damages_for_interpolation")

In [ ]:
# Define a new directory for the aggregated files by HYBAS_ID
aggregated_dir = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/joined_networks_catchments_damages_for_interpolation_by_HYBAS_ID")

## Step 1: Read in useful data

In [ ]:
# List all files in the directory, sorted alphabetically by file name
all_direct_damages_files = sorted(direct_damages_folder.iterdir(), key=lambda f: f.name)
display("All files in directory:", [f.name for f in all_direct_damages_files])

In [ ]:
# Filter out non-parquet files (e.g., .DS_Store)
parquet_files = [f for f in all_direct_damages_files if f.suffix == ".parquet"]

if not parquet_files:
    print("No parquet files found!")
else:
    # Pick the first parquet file for inspection
    example_file = parquet_files[0]
    print(f"Inspecting file: {example_file.name}")
    
    # Read the file into a DataFrame
    df_example = pd.read_parquet(example_file)
    
    # Print column names and shape
    display("Column names:", df_example.columns.tolist())
    display("DataFrame shape:", df_example.shape)
    
    # Display the first 5 rows
    display("First 5 rows:")
    display(df_example.head())
    
    # Optional: Display summary statistics for numeric columns
    display("Summary statistics:")
    display(df_example.describe())

# Pick the first parquet file
example_file = parquet_files[0]

# Read the file into a DataFrame
df_example = pd.read_parquet(example_file)

# Print the column names
display("Column names in", example_file.name, ":", df_example.columns.tolist())

In [ ]:
# Define the path to the specific parquet file
electricity_file_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/Results/direct_damages_summary/electricity_network_v3.1_edges_damages.parquet")

# Read the parquet file into a DataFrame
df = pd.read_parquet(electricity_file_path)

# Filter out columns that contain both "fluvial" and "baseline" (ignoring case)
cols_of_interest = [col for col in df.columns if "fluvial" in col.lower() and "baseline" in col.lower()]

# Display the names of the columns of interest
display("Columns containing 'fluvial' and 'baseline':", cols_of_interest)

# Display the first few rows of the filtered data
display("Preview of data in the selected columns:")
display(df[cols_of_interest])

In [ ]:
# Get a list of all files ending with _damages.parquet
parquet_files = list(direct_damages_folder.glob("*_damages.parquet"))


In [ ]:
filtered_dfs = []  # list to store filtered DataFrames

for file in parquet_files:
    df = pd.read_parquet(file)
    # Keep columns that either:
    # 1. Contain both "fluvial" and "baseline"
    # 2. Are asset ID columns (e.g., "id", "node_id", "edge_id", "osm_id", etc.)
    cols_to_keep = [
        col for col in df.columns
        if ("fluvial" in col and "baseline" in col)
           or (col.lower() == "id" or col.lower().endswith("_id"))
    ]
    filtered_df = df[cols_to_keep]
    filtered_dfs.append(filtered_df)
    print(f"{file.name}: {filtered_df.columns.tolist()}")

In [ ]:
# List all files in the directory to check that it is pointing to the right place
all_networks_catchments_files = sorted(networks_catchments_folder.iterdir(), key=lambda f: f.name)
display("All files in directory:", [f.name for f in all_networks_catchments_files])

In [ ]:
# Pick the first file from the sorted list
example_file = all_networks_catchments_files[0]

# Read the file into a GeoDataFrame
gdf = gpd.read_file(example_file)

# Print the column names
display("Column names in", example_file.name, ":", gdf.columns.tolist())

In [ ]:
for file in all_networks_catchments_files:
    try:
        gdf = gpd.read_file(file)
        print(f"{file.name} columns: {gdf.columns.tolist()}")
    except Exception as e:
        print(f"Error reading {file.name}: {e}")

In [ ]:
# Function to extract a key from the catchments file name.
# We'll assume the key is a tuple (sector, feature_type) where:
#   - sector: the first part of the file name (e.g. "airports" or "buildings")
#   - feature_type: determined by the presence of "edges", "nodes" or "areas" in the name.
def extract_catchments_key(filepath: Path):
    base = filepath.stem  # e.g. "airports_areas_catchments_intersection"
    base_lower = base.lower()
    parts = base_lower.split("_")
    sector = parts[0]  # e.g. "airports"
    if "edges" in base_lower:
        feature_type = "edges"
    elif "nodes" in base_lower:
        feature_type = "nodes"
    elif "areas" in base_lower:
        feature_type = "areas"
    else:
        feature_type = "unknown"
    return (sector, feature_type)


In [ ]:
# Function to extract a key from the damages file name.
# For instance, "airport_polygon_areas_damages.parquet" would become ("airport", "areas")
def extract_damages_key(filepath: Path):
    base = filepath.stem  # e.g. "airport_polygon_areas_damages"
    base_lower = base.lower()
    parts = base_lower.split("_")
    sector = parts[0]  # e.g. "airport"
    if "edges" in base_lower:
        feature_type = "edges"
    elif "nodes" in base_lower:
        feature_type = "nodes"
    elif "areas" in base_lower:
        feature_type = "areas"
    else:
        feature_type = "unknown"
    return (sector, feature_type)



In [ ]:
# Build a dictionary for catchments files keyed by (sector, feature_type)
catchments_dict = {}
for fp in all_networks_catchments_files:
    key = extract_catchments_key(fp)
    catchments_dict[key] = fp

# A helper function to pick the join column from a DataFrame.
# We assume the join column is one of 'node_id', 'edge_id', 'osm_id', or 'id'
def get_join_column(df):
    for candidate in ['node_id', 'edge_id', 'osm_id', 'id']:
        if candidate in df.columns:
            return candidate
    raise ValueError("No join column found in dataframe")


In [ ]:
# # Now, loop through each damages file, find its matching catchments file and merge on the id column.
# for dmg_file in all_direct_damages_files:
#     # Skip buildings files (you can adjust the condition if needed)
#     # if "buildings" in dmg_file.name.lower():
#     #     print(f"Skipping buildings file: {dmg_file.name}")
#     #     continue

#     dmg_key = extract_damages_key(dmg_file)
#     matching_key = None
#     for ckey in catchments_dict.keys():
#         dmg_sector = dmg_key[0]
#         catch_sector = ckey[0]
#         # New matching condition: check for exact match, singular/plural, or if one string starts with the other.
#         if ((dmg_sector == catch_sector) or 
#             (dmg_sector + "s" == catch_sector) or 
#             (catch_sector + "s" == dmg_sector) or
#             (catch_sector.startswith(dmg_sector)) or
#             (dmg_sector.startswith(catch_sector))) and (dmg_key[1] == ckey[1]):
#             matching_key = ckey
#             break    
    
#     if matching_key is None:
#         print(f"No matching catchments file for {dmg_file.name}")
#         continue

#     catch_file = catchments_dict[matching_key]
#     print(f"Joining {dmg_file.name} with {catch_file.name}")

#     # Read the damages file and then filter it to keep only the fluvial/baseline columns and join ID
#     df_dmg = pd.read_parquet(dmg_file)
#     cols_to_keep = [
#         col for col in df_dmg.columns
#         if ("fluvial" in col and "baseline" in col)
#            or (col.lower() == "id" or col.lower().endswith("_id"))
#     ]
#     df_dmg = df_dmg[cols_to_keep]

#     # Read the catchments file

#     gdf_catch = gpd.read_file(catch_file)

    
#     # Get the join columns from both DataFrames
#     dmg_join_col = get_join_column(df_dmg)
#     catch_join_col = get_join_column(gdf_catch)
#     print(f"  Damages join column: {dmg_join_col} | Catchments join column: {catch_join_col}")

#     # Merge the dataframes on the id column.
#     # Using a left merge (all catchments and matching damages)
#     joined = gdf_catch.merge(df_dmg, left_on=catch_join_col, right_on=dmg_join_col, how="left")
#     print("  Joined shape:", joined.shape)

#     # Save the joined output to the output_dir
#     output_path = output_dir / f"joined_{catch_file.name}"
#     joined.to_file(str(output_path), driver="GPKG")


In [ ]:
# Build aggregation dictionary and group by HYBAS_ID
cols_to_agg = [
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_20__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_50__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_amin', 
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_100__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_200__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_500__rcp_baseline__epoch_2010__conf_None_amax',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_amin',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_mean',
    'fluvial__rp_1500__rcp_baseline__epoch_2010__conf_None_amax'
]

# Loop over each .gpkg file in the folder
for gpkg_file in aggregated_dir.glob("*.gpkg"):
    print(f"Processing file: {gpkg_file.name}")
    
    # Read the file into a GeoDataFrame
    gdf = gpd.read_file(gpkg_file)
    
    # Check if both HYBAS_ID and all the required columns are in the file
    if "HYBAS_ID" in gdf.columns and all(col in gdf.columns for col in cols_to_agg):
        # Extract only the HYBAS_ID and columns to aggregate
        df = gdf[["HYBAS_ID"] + cols_to_agg]
        
        # Group by HYBAS_ID and sum the specified columns
        aggregated_summary = df.groupby("HYBAS_ID", as_index=False)[cols_to_agg].sum()
        
        # Display the aggregated summary
        display(aggregated_summary)
        display("\n" + "-"*60 + "\n")
        
        # Save the aggregated summary to a CSV file in the new folder
        output_file = aggregated_summaries / f"{gpkg_file.stem}_aggregated.csv"
        aggregated_summary.to_csv(output_file, index=False)
        display(f"Saved aggregated summary to {output_file}\n")
    else:
        display(f"Skipping {gpkg_file.name}: required columns missing.\n")

### Interpolating functions

In [ ]:
def interpolate_damages_log(rp: float, rp_l: float, value_l: np.ndarray, rp_u: float, value_u: np.ndarray) -> np.ndarray:
    """
    Logarithmically interpolate between two known return period values.
    """
    rp_factor = (np.log(rp) - np.log(rp_l)) / (np.log(rp_u) - np.log(rp_l))
    return value_l + (value_u - value_l) * rp_factor


In [ ]:
def get_rp_metric_cols(df: pd.DataFrame, metric: str) -> tuple[list[str], list[float]]:
    """
    Extract the columns and their corresponding return periods for a given metric.
    Assumes columns are in the format:
    'fluvial__rp_<rp>__rcp_baseline__epoch_2010__conf_None_<metric>'
    """
    rp_cols = [
        col for col in df.columns 
        if col.startswith("fluvial") and f"__conf_None_{metric}" in col
    ]
    # Extract the rp value from the second segment (e.g., 'rp_20')
    rps = [float(col.split("__")[1].replace("rp_", "")) for col in rp_cols]
    # Sort the pairs by rp (ascending order)
    sorted_pairs = sorted(zip(rps, rp_cols), key=lambda x: x[0])
    sorted_rps, sorted_cols = zip(*sorted_pairs)
    return list(sorted_cols), list(sorted_rps)


In [ ]:
def pick_upper_lower_rps(rp: float, rps: list[float]) -> tuple[float, float, int]:
    """
    Given a target return period and a sorted list of known rps,
    return the lower and upper bounding rps and the index of the upper bound.
    If the target is outside the known range, both bounds will be the same.
    """
    bin_index = np.searchsorted(rps, rp, side="left")
    if bin_index == 0:
        return rps[0], rps[0], 0
    elif bin_index == len(rps):
        return rps[-1], rps[-1], len(rps) - 1
    else:
        return rps[bin_index - 1], rps[bin_index], bin_index


In [ ]:
def interpolate_rp_metric(rp: float, metric: str, df: pd.DataFrame) -> pd.Series:
    """
    Interpolate values for a given target return period (rp) and metric across all rows.
    """
    rp_cols, rps = get_rp_metric_cols(df, metric)
    rp_l, rp_u, upper_idx = pick_upper_lower_rps(rp, rps)
    # If the target rp is outside the known range, just return the edge value.
    if rp_l == rp_u:
        col = rp_cols[upper_idx]
        return df[col]
    else:
        col_l = rp_cols[upper_idx - 1]
        col_u = rp_cols[upper_idx]
        values_l = df[col_l].to_numpy()
        values_u = df[col_u].to_numpy()
        interp_values = interpolate_damages_log(rp, rp_l, values_l, rp_u, values_u)
        return pd.Series(interp_values, index=df.index)


In [ ]:
def interpolate_rp_values(target_rps: list[float], metrics: list[str], df: pd.DataFrame) -> pd.DataFrame:
    """
    For each target return period and metric, interpolate the values and return a new DataFrame
    with columns following the naming pattern:
    'fluvial__rp_<target_rp>__rcp_baseline__epoch_2010__conf_None_<metric>'
    """
    # Retain HYBAS_ID if it exists
    if "HYBAS_ID" in df.columns:
        interpolated_df = df[["HYBAS_ID"]].copy()
    else:
        interpolated_df = pd.DataFrame(index=df.index)
    
    for metric in metrics:
        for rp in target_rps:
            col_name = f"fluvial__rp_{rp}__rcp_baseline__epoch_2010__conf_None_{metric}"
            interpolated_df[col_name] = interpolate_rp_metric(rp, metric, df)
    
    return interpolated_df



In [ ]:
def interpolate_rp_fluvial(target_rp: float, df: pd.DataFrame, metrics: list[str] = ['amin', 'mean', 'amax']) -> pd.DataFrame:
    """
    Interpolate the damage metrics for a given target return period (target_rp)
    from the aggregated DataFrame that has columns of the form:
    'fluvial__rp_<rp>__rcp_baseline__epoch_2010__conf_None_<metric>'
    
    Returns a new DataFrame with interpolated columns following the same naming pattern.
    """
    # Start with HYBAS_ID if it exists
    if "HYBAS_ID" in df.columns:
        out_df = df[["HYBAS_ID"]].copy()
    else:
        out_df = pd.DataFrame(index=df.index)
    
    # Loop over each metric (e.g., amin, mean, amax)
    for metric in metrics:
        # Get the known columns and their return period values for this metric
        rp_cols, rps = get_rp_metric_cols(df, metric)
        # Find the two RPs that bracket the target RP
        rp_l, rp_u, upper_idx = pick_upper_lower_rps(target_rp, rps)
        
        col_name = f"fluvial__rp_{target_rp}__rcp_baseline__epoch_2010__conf_None_{metric}"
        
        # If target is outside the known range (or exactly matches a known value), just copy the known column
        if rp_l == rp_u:
            out_df[col_name] = df[rp_cols[upper_idx]]
        else:
            col_l = rp_cols[upper_idx - 1]
            col_u = rp_cols[upper_idx]
            values_l = df[col_l].to_numpy()
            values_u = df[col_u].to_numpy()
            # Interpolate using logarithmic scaling
            interp_vals = interpolate_damages_log(target_rp, rp_l, values_l, rp_u, values_u)
            out_df[col_name] = interp_vals
    
    return out_df


### Other interpolating code

In [ ]:
# Suppose 'aggregated_summary' is your aggregated DataFrame with columns for:
# rp 20, 50, 100, 200, 500, and 1500.
# For example, you might load it from a CSV file:
# aggregated_summary = pd.read_csv("path/to/your_aggregated.csv")

# Define the full range of return periods to interpolate over, for example every integer from 20 to 1500.
rp_full = list(range(20, 1501))
metrics = ['amin', 'mean', 'amax']

# # Now, get the expanded DataFrame with interpolated values.
# expanded_df = interpolate_rp_values(rp_full, metrics, aggregated_summary)


In [ ]:
# For instance, to get the interpolated values for RP=75:
target_rp = 70.0
interpolated_values = interpolate_rp_fluvial(target_rp, aggregated_summary)

# Display the result
print(interpolated_values.head())

In [ ]:
# Define the known return periods (the ones you have columns for)
rp_known = [20, 50, 100, 200, 500, 1500]

# Define the full range to interpolate over (here every integer from 20 to 1500)
rp_full = np.arange(20, 1501)


In [ ]:
aggregated_files = list(aggregated_summaries.glob("*_aggregated.csv"))

In [ ]:
# The three damage metrics to interpolate
metrics = ['amin', 'mean', 'amax']

def interpolate_row(row):
    """
    Given a row from the aggregated summary (one HYBAS_ID),
    interpolate the damage metrics for every return period in rp_full.
    Returns a dictionary with HYBAS_ID and interpolated columns.
    """
    hybas_id = row["HYBAS_ID"]
    out = {"HYBAS_ID": hybas_id}
    for metric in metrics:
        # Build the list of known values using the original column names.
        known_vals = [row[f"fluvial__rp_{rp}__rcp_baseline__epoch_2010__conf_None_{metric}"] for rp in rp_known]
        # Interpolate over the full range.
        interpolated_vals = np.interp(rp_full, rp_known, known_vals)
        # Create a column for each return period with the same naming pattern.
        for i, rp in enumerate(rp_full):
            col_name = f"fluvial__rp_{rp}__rcp_baseline__epoch_2010__conf_None_{metric}"
            out[col_name] = interpolated_vals[i]
    return out



In [ ]:
for file in aggregated_files:
    print(f"Processing file: {file.name}")
    df_agg = pd.read_csv(file)
    
    # List to collect interpolated rows (one per HYBAS_ID)
    interpolated_data = []
    for idx, row in df_agg.iterrows():
        interp_dict = interpolate_row(row)
        interpolated_data.append(interp_dict)
    
    # Build a wide DataFrame from the list of dictionaries.
    df_expanded = pd.DataFrame(interpolated_data)
    
    # Order columns: HYBAS_ID first, then for each rp in rp_full (in order) and for each metric.
    ordered_cols = ["HYBAS_ID"]
    for rp in rp_full:
        for metric in metrics:
            col_name = f"fluvial__rp_{rp}__rcp_baseline__epoch_2010__conf_None_{metric}"
            ordered_cols.append(col_name)
    df_expanded = df_expanded[ordered_cols]
    
    # Optionally, inspect the first few columns/rows
    display(f"Expanded DataFrame for {file.name}:", df_expanded)

    # Save the expanded DataFrame to a new CSV file
    output_file = aggregated_summaries / f"{file.stem}_expanded.csv"
    df_expanded.to_csv(output_file, index=False)
    print(f"Saved expanded data to {output_file}")


In [ ]:
# I'm not sure if this is checking correctly tbh - need to have a look into it

rp_known = [20, 50, 100, 200, 500, 1500]
metrics = ['amin', 'mean', 'amax']

# Loop through each HYBAS_ID in the aggregated data
for idx, orig_row in df_agg.iterrows():
    hybas_id = orig_row["HYBAS_ID"]
    expanded_row = df_expanded[df_expanded["HYBAS_ID"] == hybas_id].iloc[0]
    for rp in rp_known:
        for metric in metrics:
            col_name = f"fluvial__rp_{rp}__rcp_baseline__epoch_2010__conf_None_{metric}"
            orig_val = orig_row[col_name]
            interp_val = expanded_row[col_name]
            if not np.isclose(orig_val, interp_val):
                print(f"Discrepancy for HYBAS_ID {hybas_id}, {col_name}: original {orig_val}, interpolated {interp_val}")
            else:
                print(f"Match for HYBAS_ID {hybas_id}, {col_name}")